# 06-02 AutoGen 工具注册与调用

AutoGen 中工具注册到 Agent，Agent 在对话中决定是否调用。

**本节目标**：工具注册、Executor Agent、工具调用流程

---

In [ ]:
import os, sys, json
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")

try:
    from autogen import ConversableAgent, register_function
    HAS_AG = True
except ImportError:
    HAS_AG = False
    print("AutoGen 未安装")

## 1. 定义并注册工具

In [ ]:
AD_DB = {
    "ad_1": {"title": "游戏皮肤", "impressions": 50000, "clicks": 1500, "cost": 750, "converts": 120},
    "ad_2": {"title": "美妆新品", "impressions": 30000, "clicks": 600, "cost": 480, "converts": 45},
}

def get_ad_stats(ad_id: str) -> str:
    """获取广告的投放效果数据"""
    data = AD_DB.get(ad_id)
    if not data:
        return f"未找到广告 {ad_id}"
    data["ctr"] = round(100 * data["clicks"] / data["impressions"], 2)
    return json.dumps(data, ensure_ascii=False)

def check_compliance(text: str) -> str:
    """检查广告文案合规性"""
    issues = [w for w in ["最", "第一", "绝对"] if w in text]
    return json.dumps({"ok": len(issues) == 0, "issues": issues}, ensure_ascii=False)

if HAS_AG:
    llm_config = {"config_list": [{"model": "gpt-4o-mini", "api_key": os.environ.get("OPENAI_API_KEY", "")}]}
    
    # 调用者 Agent（决定调哪个工具）
    caller = ConversableAgent(
        name="analyst",
        system_message="你是广告分析师，使用工具获取数据并给出建议。",
        llm_config=llm_config,
    )
    
    # 执行者 Agent（执行工具调用）
    executor = ConversableAgent(
        name="executor",
        llm_config=False,  # 不需要 LLM，纯执行
        human_input_mode="NEVER",
    )
    
    # 注册工具
    register_function(
        get_ad_stats,
        caller=caller,      # 谁可以调用
        executor=executor,   # 谁来执行
        description="获取广告投放效果数据",
    )
    register_function(
        check_compliance,
        caller=caller,
        executor=executor,
        description="检查广告文案合规性",
    )
    
    print("工具注册完成")
    print(f"  caller({caller.name}) 可以决定调用工具")
    print(f"  executor({executor.name}) 负责执行工具")
else:
    print("""
AutoGen 工具注册:
  register_function(
      func,                # Python 函数
      caller=agent_a,      # 决定调用的 Agent
      executor=agent_b,    # 执行调用的 Agent
      description="...",   # 工具描述
  )

流程: caller 决定调用 → executor 执行 → 结果返回给 caller
    """)

In [ ]:
# 发起带工具调用的对话
if HAS_AG and os.environ.get("OPENAI_API_KEY"):
    result = executor.initiate_chat(
        caller,
        message="请查看 ad_1 的数据并给出优化建议",
        max_turns=3,
    )
    print("\n最终建议:")
    print(result.chat_history[-1]["content"][:200])
else:
    print("模拟工具调用流程:")
    print(f"  1. caller 决定调用 get_ad_stats('ad_1')")
    print(f"  2. executor 执行: {get_ad_stats('ad_1')}")
    print(f"  3. caller 根据数据给出建议")

## 面试速记

| 问题 | 要点 |
|------|------|
| AutoGen 工具调用 vs LangChain | AutoGen: caller/executor 分离；LangChain: bind_tools 到单个 Agent |
| 为什么分 caller/executor | 安全考虑：决策和执行分离，executor 可以加权限控制 |

**下一节**: `03_multi_agent_conversation.ipynb`